# Analyze

One saved run against its two reference points. Name a topic and a stem; the run's own record says which topic it answered and how it was built, so everything below is derived: the country and per-region comparison against the *actual* result and the *claimed* votes, covariance and correlation between options, the spread of the per-subject distributions, every run of the topic side by side, and the latent runs by area. Derived tables go to `results/<topic>/analysis/` (regenerable), figures to `graphs/`.

Legacy runs are keyed by row position and were built from the frozen `legacy_clean_data.csv`; a run saved by `execute` is keyed by subject id and joins its dataset's profile. The record tells them apart.

In [ ]:
import pandas as pd

from panel import paths
from panel.analysis import aggregate, compare
from panel.analysis.groups import claimed_by, predicted_by
from panel.analysis.plots import comparison, distributions, matrices
from panel.dataset import load_claims, load_dataset, read_profile
from panel.run import ResultStore
from panel.topic import load_topic

In [ ]:
# Pick a saved run. STORE.runs() lists every stem of the topic.

TOPIC_NAME = "election_2021_legacy"
STEM = "sod_OpenAI_gpt-4o-mini_VProb_t=0.0"

STORE = ResultStore(paths.RESULTS_DIR, TOPIC_NAME)
RECORD = STORE.record(STEM)
TOPIC = load_topic(RECORD["spec"]["topic"])
OPTIONS = list(TOPIC.options)
TOTAL = TOPIC.reference.total
MODEL = RECORD["spec"]["model"]

LEGACY = "description" not in RECORD["spec"]
DATASET = load_dataset("sod" if LEGACY else RECORD["spec"]["dataset"])
PROFILE = pd.read_csv(paths.DATA_DIR / "legacy_clean_data.csv", index_col="id") if LEGACY else read_profile(DATASET)
CLAIMS = load_claims(DATASET, TOPIC.name)

results = TOPIC.rename(STORE.frame(STEM, normalized=True))
reference = TOPIC.load_reference().set_index(TOPIC.reference.key)
paths.GRAPHS_DIR.mkdir(exist_ok=True)
print(f"{STEM}: {len(results)} rows on topic {TOPIC.name}")

## Basic statistics

In [ ]:
results[list(TOPIC.turnout) + OPTIONS].describe()

In [ ]:
VOTED, NOT_VOTED = TOPIC.turnout
voters = results[VOTED] > 0
print(f"attendance probabilities summing to 1: {(results[list(TOPIC.turnout)].sum(axis=1) >= 1).mean():.2%}")
print(f"certain non-voters, P({VOTED}) = 0: {(~voters).mean():.2%}")
print(f"no option preference, options summing to 0: {(results[OPTIONS].sum(axis=1) <= 0.001).mean():.2%}")

## Country level

In [ ]:
predicted = aggregate.aggregate_results_from_df(results, TOPIC)
claimed = aggregate.claimed_distribution(PROFILE, CLAIMS, TOPIC)
actual = reference.loc[TOTAL, TOPIC.columns]
pd.DataFrame({"predicted": predicted, "claimed": claimed, "actual": actual}).round(4)

In [ ]:
fig = comparison.distribution_bars(predicted, actual, OPTIONS, TOPIC.turnout, reference_label="Actual",
                                   labels=TOPIC.names, title=f"{MODEL}: predicted vs actual")
fig.savefig(paths.GRAPHS_DIR / f"{STEM}_vs_actual.png", bbox_inches="tight")
fig = comparison.distribution_bars(predicted, claimed, OPTIONS, TOPIC.turnout, reference_label="Claimed",
                                   labels=TOPIC.names, title=f"{MODEL}: predicted vs claimed")
fig.savefig(paths.GRAPHS_DIR / f"{STEM}_vs_claimed.png", bbox_inches="tight")

## Covariance and correlation between options

In [ ]:
covariance = results[OPTIONS].cov()
correlation = results[OPTIONS].corr()
covariance.to_csv(STORE.analysis(f"cov_matrix_{STEM}.csv"))
correlation.to_csv(STORE.analysis(f"corr_matrix_{STEM}.csv"))
fig = matrices.heatmap(covariance, title=f"Covariance - {MODEL}", labels=TOPIC.names)
fig.savefig(paths.GRAPHS_DIR / f"cov_matrix_{STEM}.png", bbox_inches="tight")
fig = matrices.heatmap(correlation, title=f"Correlation - {MODEL}", vmin=-1, vmax=1, labels=TOPIC.names)
fig.savefig(paths.GRAPHS_DIR / f"corr_matrix_{STEM}.png", bbox_inches="tight")

## By region

Predicted, claimed and actual per region, the whole country as the total row, and the MAE between each pair.

In [ ]:
KEY = "region"
sources = {
    "Predicted": predicted_by(results, PROFILE, KEY, TOPIC, total=TOTAL),
    "Claimed": claimed_by(PROFILE, KEY, CLAIMS, TOPIC, total=TOTAL),
    "Actual": reference,
}
sources = {name: frame.loc[reference.index] for name, frame in sources.items()}
regional_mae = compare.pairwise(sources, OPTIONS, pairs=[("Predicted", "Actual"), ("Predicted", "Claimed"), ("Actual", "Claimed")], decimals=3)
regional_mae.to_csv(STORE.analysis(f"regional_pairwise_mae_{STEM}.csv"))
fig = matrices.error_grid(regional_mae, ["Predicted", "Claimed", "Actual"], ncols=5, title=f"MAE by region - {MODEL}")
fig.savefig(paths.GRAPHS_DIR / f"regional_mae_{STEM}.png", bbox_inches="tight")
regional_mae

## Spread of the per-subject distributions

In [ ]:
fig = distributions.option_distribution(results, OPTIONS, title=f"Option probabilities per subject - {MODEL}",
                                        kind="violin", add_points=False, labels=TOPIC.names)
fig.savefig(paths.GRAPHS_DIR / f"option_distribution_{STEM}.png", bbox_inches="tight")
fig = distributions.spread_histograms(results, OPTIONS, title=f"Distance from the mean subject - {MODEL}")

## Every run of this topic with the same response shape

In [ ]:
SHAPE = RECORD["spec"].get("response_model", RECORD["spec"].get("shape"))
same_shape = [stem for stem in STORE.runs() if STORE.record(stem)["spec"].get("response_model", STORE.record(stem)["spec"].get("shape")) == SHAPE]
by_run = pd.DataFrame({stem: aggregate.aggregate_results_from_df(TOPIC.rename(STORE.frame(stem)), TOPIC) for stem in same_shape}).T
by_run["MAE vs actual"] = [compare.distances(actual[OPTIONS], row[OPTIONS])["MAE"] for _, row in by_run.iterrows()]
by_run["MAE vs claimed"] = [compare.distances(claimed[OPTIONS], row[OPTIONS])["MAE"] for _, row in by_run.iterrows()]
fig = comparison.bars_by_source(by_run["MAE vs actual"], "MAE vs actual (%)", scale=100, title=f"{TOPIC.name}: runs against the actual result")
by_run.round(4)

## Latent runs by area

Runs on the `regions` dataset ask about a place instead of a person; each area is aggregated over its repeats. The legacy topic also holds pre-RunSpec tables of the same kind under `preferences/`.

In [ ]:
latent = {}
for stem in STORE.runs():
    if STORE.record(stem)["spec"].get("dataset") == "regions":
        areas = read_profile(load_dataset("regions"))
        latent[stem] = predicted_by(TOPIC.rename(STORE.frame(stem)), areas, "region", TOPIC)
        latent[stem].to_csv(STORE.analysis(f"by_region_{stem}.csv"))
if STORE.path("preferences").exists():
    for path in sorted(STORE.path("preferences").glob("by_region_latent_*.csv")):
        latent[path.stem[len("by_region_"):]] = TOPIC.rename(pd.read_csv(path, index_col=0)).set_index("region")
for name, by_area in latent.items():
    fig = comparison.distribution_bars(by_area.loc[TOTAL], actual, OPTIONS, TOPIC.turnout, reference_label="Actual",
                                       labels=TOPIC.names, title=f"{name}: {TOTAL}")
    fig.savefig(paths.GRAPHS_DIR / f"latent_{name}.png", bbox_inches="tight")
print(f"{len(latent)} latent runs")